In [1]:
import os
import gc
import json
import random
import pickle as pkl
import networkx as nx
import pandas as pd
import numpy as np
import xgboost as xgb
from tqdm import tqdm
from sklearn import metrics
from scipy.optimize import minimize_scalar
from fastnode2vec import Node2Vec, Graph
from utils import get_texts_for_ptype_pmid, get_embeddings, get_texts_citations_labels

from ray.tune.search import ConcurrencyLimiter
from ray.tune.search.bayesopt import BayesOptSearch
from ray import tune, train
from ray.train import Checkpoint
from custom_models.custom_model import BatchVectorizerXGBoostConcat

/home/gregory_kell/study_design_classifier/study-design-classifier/.venv_gnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-11-26 07:44:52,555	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-11-26 07:44:52,646	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [5]:
    results_file = "results_node2vec_gbdt_sample.json"
    bow_node2vec_gbdt_concat_metrics_adjusted_results_path = "results_adjusted_metrics_bow_node2vec_gbdt_concat.json"
    MODEL_DIR = "../data"
    EMBEDDING_FILENAME = os.path.join(MODEL_DIR, 'node_model.pkl')
    PMIDS = '../data/pmids.pkl'  
    
    with open(results_file, "r") as f:
        results_list = json.load(f)
        
    with open("../data/graph_lst.pkl", "rb") as f:
        graph_lst = pkl.load(f)

    graph = Graph(graph_lst, directed=False, weighted=False)

    node2vec = Node2Vec(graph, dim=10, walk_length=100, window=10, p=2.0, q=0.5, workers=5)  # Use temp_folder for big graphs


In [6]:
model = node2vec.wv.load_word2vec_format(EMBEDDING_FILENAME)

In [7]:
with open("../data/pmid_to_openAlex.json") as f:
    pmid_to_openAlex = json.load(f)
    
with open(PMIDS, 'rb') as f:
    pmids = pkl.load(f)

with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
    prop_typ_id_dict = pkl.load(f)

ptypes = prop_typ_id_dict.keys()

In [8]:
def predict(df, db=0):
    return [int(x>=db) for x in df ]

In [9]:
def get_scores(predictions):
    return [np.exp(p[1])/(np.exp(p[0]) + np.exp(p[1])) for p in predictions]

In [10]:
neg_pos_ratio_dict = {
    "is_rct": 5585042/489295,
    "is_sr": 5659292/121063,
    "is_casecontrol": 5628219/65790,
    "is_cohort": 5632460/78688,
    "is_guideline": 5676411/33826
}

def calc_precision(recall, specificity, neg_pos_ratio):
    return recall / (recall + neg_pos_ratio * (1 - specificity) + 1e-15)

def calc_f1(recall, precision):
    return 2*recall*precision/(recall+precision + 1e-15)

In [11]:
metrics_lst = []
for results_json in tqdm(results_list):
    ptype = results_json["ptype"]
    pos_train_embeddings, neg_train_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='train')
    pos_val_embeddings, neg_val_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='val')
    pos_test_embeddings, neg_test_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='test')
        
    pos_train_pmids = list(pos_train_embeddings.keys())
    neg_train_pmids = list(neg_train_embeddings.keys())
    
    pos_val_pmids = list(pos_val_embeddings.keys())
    neg_val_pmids = list(neg_val_embeddings.keys())
    pos_test_pmids = list(pos_test_embeddings.keys())
    neg_test_pmids = list(neg_test_embeddings.keys())
    pmids_to_use = {
        "pos_train": pos_train_pmids,
        "neg_train": neg_train_pmids,
        "pos_val": pos_val_pmids,
        "neg_val": neg_val_pmids,
        "pos_test": pos_test_pmids,
        "neg_test": neg_test_pmids
    }
        
    train_set, val_set, test_set = get_texts_for_ptype_pmid(ptype, inputs="titles_abstracts", neg_to_pos_ratio=1, reset_index=True)
        
    train_set_concat = {}
    for pmid in pos_train_pmids:
        train_set_concat[pmid] = {}
        train_set_concat[pmid]["X_graph"] = pos_train_embeddings[pmid]
        train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
        train_set_concat[pmid]["y"] = 1

    for pmid in neg_train_pmids:
        train_set_concat[pmid] = {}
        train_set_concat[pmid]["X_graph"] = neg_train_embeddings[pmid]
        train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
        train_set_concat[pmid]["y"] = 0

    test_set_concat = {}
    for pmid in pos_test_pmids:
        test_set_concat[pmid] = {} 
        test_set_concat[pmid]["X_graph"] = pos_test_embeddings[pmid]
        test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
        test_set_concat[pmid]["y"] = 1

    for pmid in neg_test_pmids:
        test_set_concat[pmid] = {}
        test_set_concat[pmid]["X_graph"] = neg_test_embeddings[pmid]
        test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
        test_set_concat[pmid]["y"] = 0
        
    train_df = get_texts_citations_labels(train_set_concat)
    train_df = train_df.sample(frac=1)
        
    test_df = get_texts_citations_labels(test_set_concat)
    test_df = test_df.sample(frac=1)
        
        
    bvc = BatchVectorizerXGBoostConcat(
        objective="binary:logistic",
            # max_depth=config["max_depth"],
            max_depth=18,
            gamma=results_json["gamma"],
            reg_alpha=results_json["reg_alpha"],
            reg_lambda=results_json["reg_lambda"],
            colsample_bytree=results_json["colsample_bytree"],
            min_child_weight=results_json["min_child_weight"],
            # n_estimators=config["n_estimators"],
            n_estimators=180,
            seed=0,
        )
        
    bvc.fit(train_df["texts"], train_df["node_embeds"], train_df["labels"])
    predictions = bvc.predict_proba(test_df["texts"], test_df["node_embeds"])
    
    scores = get_scores(predictions)
    y_test = test_df["labels"]
        
    def find_db(db):
        y_pred_test = predict(scores, db)
        recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
        return abs(recall_test - 0.985)

    auc_roc = metrics.roc_auc_score(y_test, scores)

    res = minimize_scalar(find_db)
    db = res["x"]
    y_pred_test = predict(scores, db)
    
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
    precision_test = calc_precision(recall_test, spec_test, neg_pos_ratio_dict[ptype])
    f1_test = calc_f1(recall_test, precision_test)
    
    print('y_test shape:', y_test.shape)
    print('logits shape:', predictions.shape)
    
    metrics_dict = {
        "ptype": ptype,
        "y_test": y_test,
        "logits": predictions,
        "db": db,
        "scores": scores,
        "auc_roc": auc_roc,
        "recall_test": recall_test,
        "spec_test": spec_test,
        "precision_test": precision_test,
        "f1_test": f1_test
    }
    metrics_lst.append(metrics_dict)
    
    if os.path.exists(bow_node2vec_gbdt_concat_metrics_adjusted_results_path):
        with open(bow_node2vec_gbdt_concat_metrics_adjusted_results_path, "rb") as f:
            metrics_lst = pkl.load(f)
    
    metrics_lst.append(metrics_dict)
    with open(bow_node2vec_gbdt_concat_metrics_adjusted_results_path, "wb") as f:
        pkl.dump(metrics_lst, f)
        
    del bvc
    gc.collect()

100%|██████████| 4000/4000 [00:00<00:00, 182022.72it/s]
/home/gregory_kell/study_design_classifier/study-design-classifier/.venv_gnn/lib/python3.11/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
/home/gregory_kell/study_design_classifier/study-design-classifier/.venv_gnn/lib/python3.11/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:45,  1.93s/it]
57it [02:31,  2.65s/it]


y_test shape: (7406,)
logits shape: (7406, 2)


100%|██████████| 4000/4000 [00:00<00:00, 130136.64it/s]
/home/gregory_kell/study_design_classifier/study-design-classifier/.venv_gnn/lib/python3.11/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
21it [00:53,  2.57s/it]
 20%|██        | 1/5 [06:41<26:46, 401.62s/it]


KeyboardInterrupt: 